# Campaign Effectiveness Analyzer
## End-to-End ML Pipeline · BrightHut / Lighthouse Sanctuary

---

## 1. Problem Framing

### Business Problem

BrightHut runs periodic fundraising campaigns — concentrated bursts of outreach activity intended to drive donations — but the founders are not sure which ones actually move the needle versus those that just generate noise. They have limited staff, no dedicated marketing team, and a constrained budget for outreach. Every campaign costs real time and energy, and decisions about what to do next are currently made on intuition rather than evidence.

**The business questions are:**
1. *Which campaign attributes (timing, duration, platform mix, content type, messaging) are most associated with higher total donation revenue?* (Explanatory)
2. *Can we predict whether a planned campaign configuration will generate above-average revenue, so the org can adjust before launching?* (Predictive)
3. *Are campaigns generating new donors, or only re-engaging existing ones?* (Supplemental descriptive)

This pipeline is primarily **explanatory** — the organization needs to understand *why* some campaigns outperform others so it can systematically improve. The predictive model adds operational value by flagging planned campaigns that are configured similarly to historical underperformers.

### Who Cares About This

| Stakeholder | How they use this model |
|---|---|
| Executive Director | Annual strategy: which campaign types to prioritize and which to drop |
| Fundraising/Outreach Staff | Pre-launch checklist: "Is this campaign configured for success?" |
| Social Media Manager | Understand whether social content within campaigns actually converts to revenue |
| Board / Donors | Evidence-based reporting on fundraising efficiency |

### Prediction vs. Explanation — Explicit Choice

This pipeline delivers **both** a predictive model and an explanatory model:

- **Explanatory goal (primary):** Use OLS/Ridge regression on log-campaign-revenue to produce interpretable coefficient estimates with confidence intervals. The question is: *holding other attributes constant, how much does posting during holiday season, including video content, or using multiple platforms change expected campaign revenue?* Coefficient interpretability and statistical defensibility are the standards here — not marginal accuracy.

- **Predictive goal (secondary):** Binary classification of `high_revenue_campaign = 1` using gradient-boosted trees, scored before launch. ROC-AUC on held-out campaigns is the primary success metric for this model.

As the textbook warns: **confusing these goals is a common source of costly analytical mistakes.** We maintain two separate model objects and interpret each only within its appropriate scope.

### Defining a "Campaign"

The dataset may or may not include explicit campaign identifiers. This pipeline handles both cases:

- **Case A (explicit campaigns):** If `social_media_posts` contains a campaign ID or campaign flag column, each unique campaign ID becomes one observation.
- **Case B (temporal clustering):** If no explicit campaign ID exists, we use a sliding-window approach: a "campaign" is defined as any period in which posting frequency is at least 2× the baseline monthly rate, and we measure donation lift within a 14-day window following the campaign start. This approximates how real campaigns are structured.

Each campaign becomes **one row** in the model input; the pipeline aggregates post-level data to campaign-level features.

### Target Variable

**Explanatory model (continuous):** `log_campaign_revenue` — natural log of (total donations received during the campaign window + $1). Log-transformation stabilizes variance and makes OLS residual normality more defensible.

**Predictive model (binary):** `high_revenue_campaign = 1` if the campaign's total revenue is at or above the 60th percentile of all campaigns. This threshold focuses attention on campaigns that meaningfully outperform the median.

### Success Metrics

| Metric | Rationale |
|---|---|
| Adjusted R² (OLS) | How much revenue variance do campaign attributes explain? |
| Coefficient p-values | Are effect size estimates statistically defensible? |
| ROC-AUC (GBT) | Out-of-sample discriminative power on unseen campaigns |
| Precision/Recall @ threshold | Operational quality of go/no-go campaign recommendations |

**Error cost asymmetry:** A false positive (running a campaign predicted as high-revenue that underperforms) wastes staff time and may erode audience trust. A false negative (not running a campaign that would have succeeded) costs missed revenue. We tune the threshold modestly toward precision here — **unlike** the resident welfare pipelines, staff time is the scarce resource in campaign planning.

### Important Limitations

Campaign-level analysis is limited by sample size — even an organization running 12 campaigns per year will have few observations after a year of data. Results are directional and hypothesis-generating. External confounders (news cycles, economic conditions, platform algorithm changes) are not captured. Treat coefficient estimates as approximate ranges rather than precise causal claims.

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
# ── Install / import ─────────────────────────────────────────────────────────
# Uncomment the line below if running in a fresh Colab / minimal environment
# !pip install scikit-learn pandas numpy matplotlib seaborn requests joblib statsmodels

import warnings, json, re
warnings.filterwarnings('ignore')

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import joblib
from pathlib import Path

from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LogisticRegression, Ridge
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics         import (
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, fbeta_score, precision_recall_curve
)
from sklearn.inspection      import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid', palette='muted')
print('Imports OK')

In [ ]:
# ── Load tables from the BrightHut REST API ───────────────────────────────────
# All data is sourced from the live deployed backend — no local CSV files.
# Endpoint pattern: GET /api/tables/{tableName}  (no auth required for reads)

API_BASE = 'https://brighthut-befxhqfdabcpfscu.centralus-01.azurewebsites.net'

def load_table(name: str) -> pd.DataFrame:
    """Fetch an entire table from the BrightHut REST API."""
    url = f'{API_BASE}/api/tables/{name}'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json())
    print(f'  {name:40s}  {df.shape[0]:>5} rows  {df.shape[1]:>3} cols')
    return df

def find_col(df: pd.DataFrame, *keywords) -> str | None:
    """Return the first column name containing any of the keywords (case-insensitive)."""
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in col.lower():
                return col
    return None

print('Loading tables...')
posts_raw   = load_table('social_media_posts')
donations   = load_table('donations')
supporters  = load_table('supporters')
allocations = load_table('donation_allocations')
inkind      = load_table('in_kind_donation_items')
print('Done.')

In [ ]:
# ── Schema inspection ─────────────────────────────────────────────────────────
print('=== social_media_posts columns ===')
print(posts_raw.dtypes)
print('\n=== donations columns ===')
print(donations.dtypes)
print('\n=== supporters columns ===')
print(supporters.dtypes)
print('\nSample posts row:')
posts_raw.head(2)

In [ ]:
# ── Resolve dynamic column names ─────────────────────────────────────────────

# social_media_posts
col_post_id      = find_col(posts_raw, 'post_id', 'id')
col_platform     = find_col(posts_raw, 'platform')
col_content_type = find_col(posts_raw, 'content_type', 'type')
col_post_date    = find_col(posts_raw, 'post_date', 'date', 'posted')
col_caption      = find_col(posts_raw, 'caption', 'text', 'content', 'message', 'body')
col_likes        = find_col(posts_raw, 'likes', 'like')
col_comments     = find_col(posts_raw, 'comments', 'comment')
col_shares       = find_col(posts_raw, 'shares', 'share', 'retweet')
col_campaign     = find_col(posts_raw, 'campaign_id', 'campaign', 'is_campaign')

# donations
col_don_id       = find_col(donations, 'donation_id', 'id')
col_don_date     = find_col(donations, 'donation_date', 'date', 'created')
col_don_amount   = find_col(donations, 'amount')
col_sup_id_don   = find_col(donations, 'supporter_id', 'donor_id', 'supporter')

# supporters
col_sup_id       = find_col(supporters, 'supporter_id', 'id')
col_sup_type     = find_col(supporters, 'supporter_type', 'type')
col_sup_date     = find_col(supporters, 'join_date', 'created_date', 'date')

print('Resolved columns:')
for alias, actual in [
    ('platform', col_platform), ('content_type', col_content_type),
    ('post_date', col_post_date), ('campaign', col_campaign),
    ('likes', col_likes), ('comments', col_comments), ('shares', col_shares),
    ('donation_date', col_don_date), ('amount', col_don_amount),
    ('supporter_id (don)', col_sup_id_don), ('supporter_type', col_sup_type),
]:
    print(f'  {alias:22s} → {actual}')

In [ ]:
# ── Parse dates ───────────────────────────────────────────────────────────────
posts = posts_raw.copy()

if col_post_date:
    posts['post_date_parsed'] = pd.to_datetime(posts[col_post_date], errors='coerce')
    posts = posts.dropna(subset=['post_date_parsed'])
    posts = posts.sort_values('post_date_parsed').reset_index(drop=True)
else:
    posts['post_date_parsed'] = pd.NaT

if col_don_date:
    donations['don_date_parsed'] = pd.to_datetime(donations[col_don_date], errors='coerce')
    donations = donations.dropna(subset=['don_date_parsed'])
    donations = donations.sort_values('don_date_parsed').reset_index(drop=True)
else:
    donations['don_date_parsed'] = pd.NaT

if col_don_amount:
    donations['amount_num'] = pd.to_numeric(donations[col_don_amount], errors='coerce').fillna(0)
else:
    donations['amount_num'] = 0

print(f'Posts parsed:     {len(posts)}')
print(f'Donations parsed: {len(donations)}')
if len(donations) > 0:
    print(f'Donation date range: {donations["don_date_parsed"].min().date()} → {donations["don_date_parsed"].max().date()}')
    print(f'Total revenue in dataset: ${donations["amount_num"].sum():,.0f}')

In [ ]:
# ── Campaign identification strategy ─────────────────────────────────────────
#
# CASE A: explicit campaign ID column exists in posts_raw
# CASE B: no explicit campaign — infer from posting bursts (2× baseline rate)
#
# We detect which case applies and build a campaign_id column accordingly.

CAMPAIGN_WINDOW_DAYS = 14   # revenue attribution window after campaign start
BURST_MULTIPLIER     = 1.5  # posts-per-week must be >= this × baseline to be a campaign week

def assign_explicit_campaigns(df, campaign_col):
    """Case A: use the existing campaign column."""
    result = df.copy()
    result['campaign_id'] = result[campaign_col].astype(str).str.strip()
    result['is_campaign_post'] = (result['campaign_id'].notna() &
                                  (result['campaign_id'] != '') &
                                  (result['campaign_id'].str.lower() != 'nan')).astype(int)
    return result

def assign_burst_campaigns(df):
    """Case B: define campaigns as weeks with posting frequency >= BURST_MULTIPLIER * baseline."""
    result = df.copy()
    result['week'] = result['post_date_parsed'].dt.to_period('W')
    weekly_counts = result.groupby('week').size()
    baseline_rate = weekly_counts.median()
    burst_weeks   = weekly_counts[weekly_counts >= BURST_MULTIPLIER * baseline_rate].index
    result['is_campaign_post'] = result['week'].isin(burst_weeks).astype(int)

    # Assign a sequential campaign ID to each contiguous burst period
    result = result.sort_values('post_date_parsed').reset_index(drop=True)
    campaign_id_counter = 0
    ids = []
    prev_burst = False
    for is_camp in result['is_campaign_post']:
        if is_camp and not prev_burst:
            campaign_id_counter += 1
        ids.append(f'campaign_{campaign_id_counter:03d}' if is_camp else None)
        prev_burst = bool(is_camp)
    result['campaign_id'] = ids
    return result

# Apply the appropriate strategy
if col_campaign and col_campaign in posts.columns:
    posts = assign_explicit_campaigns(posts, col_campaign)
    strategy = 'Case A: explicit campaign column'
else:
    posts = assign_burst_campaigns(posts)
    strategy = 'Case B: inferred from posting bursts'

n_campaigns = posts['campaign_id'].nunique() - (1 if None in posts['campaign_id'].values else 0)
print(f'Campaign identification strategy: {strategy}')
print(f'Campaigns identified: {n_campaigns}')
print(f'Campaign posts: {posts["is_campaign_post"].sum()} / {len(posts)}')

In [ ]:
# ── Build campaign-level dataset ──────────────────────────────────────────────
# Each campaign becomes one observation with aggregated post features + donation outcome.

# Engagement weights
for col in [col_likes, col_comments, col_shares]:
    if col and col in posts.columns:
        posts[col] = pd.to_numeric(posts[col], errors='coerce').fillna(0)

likes_v    = posts[col_likes].values    if col_likes    else np.zeros(len(posts))
comments_v = posts[col_comments].values if col_comments else np.zeros(len(posts))
shares_v   = posts[col_shares].values   if col_shares   else np.zeros(len(posts))
posts['post_engagement'] = likes_v + (2 * comments_v) + (3 * shares_v)

# Caption features
if col_caption and col_caption in posts.columns:
    text = posts[col_caption].astype(str).fillna('')
    posts['has_cta']          = text.str.lower().str.contains(
        r'donate|support|give|help|join|share|link in bio|click').astype(int)
    posts['mentions_impact']  = text.str.lower().str.contains(
        r'resident|girl|survivor|education|safehouse|outcome|progress|transform').astype(int)
    posts['mentions_amount']  = text.str.contains(r'\$|\d+%|\d+ (girls|residents|families)').astype(int)
    posts['has_matching']     = text.str.lower().str.contains(
        r'match|double|triple|matched giving').astype(int)
    posts['caption_length']   = text.str.len()
    posts['hashtag_count']    = text.str.count(r'#\w+')
else:
    for c in ['has_cta','mentions_impact','mentions_amount','has_matching',
              'caption_length','hashtag_count']:
        posts[c] = 0

# Media type
col_media = find_col(posts, 'media_type', 'media', 'format')
if col_media and col_media in posts.columns:
    posts['has_video'] = posts[col_media].astype(str).str.lower().str.contains(
        'video|reel|clip', na=False).astype(int)
    posts['has_image'] = posts[col_media].astype(str).str.lower().str.contains(
        'image|photo|picture|img', na=False).astype(int)
else:
    posts['has_video'] = 0
    posts['has_image'] = 0

# Platform
if col_platform:
    posts['platform_clean'] = posts[col_platform].astype(str).str.lower().str.strip()
else:
    posts['platform_clean'] = 'unknown'

# ── Aggregate to campaign level ───────────────────────────────────────────────
campaign_posts = posts.dropna(subset=['campaign_id']).copy()

def campaign_agg(df):
    """Aggregate post-level features to campaign level."""
    return pd.Series({
        'campaign_start':       df['post_date_parsed'].min(),
        'campaign_end':         df['post_date_parsed'].max(),
        'num_posts':            len(df),
        'num_platforms':        df['platform_clean'].nunique(),
        'avg_engagement':       df['post_engagement'].mean(),
        'total_engagement':     df['post_engagement'].sum(),
        'max_engagement':       df['post_engagement'].max(),
        'pct_cta_posts':        df['has_cta'].mean(),
        'pct_impact_posts':     df['mentions_impact'].mean(),
        'pct_amount_posts':     df['mentions_amount'].mean(),
        'has_matching_post':    int(df['has_matching'].any()),
        'pct_video_posts':      df['has_video'].mean(),
        'pct_image_posts':      df['has_image'].mean(),
        'avg_caption_length':   df['caption_length'].mean(),
        'avg_hashtags':         df['hashtag_count'].mean(),
        'dominant_platform':    df['platform_clean'].mode().iloc[0] if len(df) > 0 else 'unknown',
    })

campaigns = campaign_posts.groupby('campaign_id').apply(campaign_agg).reset_index()

# Campaign duration
campaigns['campaign_duration_days'] = (
    (campaigns['campaign_end'] - campaigns['campaign_start'])
    .dt.total_seconds() / 86400
).clip(lower=1)

print(f'Campaign-level dataset: {len(campaigns)} campaigns')
print(campaigns[['campaign_id','campaign_start','num_posts','num_platforms','avg_engagement']].head(10))

In [ ]:
# ── Revenue attribution: link donations to campaigns ──────────────────────────
# For each campaign, sum all donations received within CAMPAIGN_WINDOW_DAYS of
# the campaign start date. Also count new donors (first-ever donation during window).

# Identify each supporter's first-ever donation date
if col_sup_id_don and col_sup_id_don in donations.columns:
    first_donation = (
        donations.groupby(col_sup_id_don)['don_date_parsed'].min()
        .rename('first_donation_date')
    )
    donations = donations.merge(first_donation, on=col_sup_id_don, how='left')
    donations['is_new_donor'] = (
        donations['don_date_parsed'] == donations['first_donation_date']
    ).astype(int)
else:
    donations['is_new_donor'] = 0

def attribute_revenue(campaign_start, window_days=CAMPAIGN_WINDOW_DAYS):
    """Sum donations and count new donors in the attribution window."""
    if pd.isnull(campaign_start):
        return 0.0, 0
    window_end = campaign_start + pd.Timedelta(days=window_days)
    mask = (
        (donations['don_date_parsed'] >= campaign_start) &
        (donations['don_date_parsed'] <= window_end)
    )
    window_don = donations[mask]
    revenue   = float(window_don['amount_num'].sum())
    new_donors = int(window_don['is_new_donor'].sum())
    return revenue, new_donors

revenues, new_donors_list = zip(*[
    attribute_revenue(row['campaign_start'])
    for _, row in campaigns.iterrows()
]) if len(campaigns) > 0 else ([], [])

campaigns['campaign_revenue']   = list(revenues)
campaigns['new_donors_acquired'] = list(new_donors_list)

print('Revenue attribution complete:')
print(campaigns[['campaign_id','campaign_revenue','new_donors_acquired']].to_string(index=False))
print(f'\nTotal attributed revenue: ${sum(revenues):,.0f}')

In [ ]:
# ── Add temporal / seasonality features ──────────────────────────────────────

campaigns['campaign_month']        = campaigns['campaign_start'].dt.month
campaigns['campaign_quarter']      = campaigns['campaign_start'].dt.quarter
campaigns['is_holiday_season']     = campaigns['campaign_month'].isin([11, 12]).astype(int)
campaigns['is_giving_tuesday']     = (
    (campaigns['campaign_month'] == 11) &
    (campaigns['campaign_start'].dt.day.between(27, 30))
).astype(int)
campaigns['is_year_end']           = (campaigns['campaign_month'] == 12).astype(int)
campaigns['is_summer']             = campaigns['campaign_month'].isin([6, 7, 8]).astype(int)
campaigns['day_of_week_start']     = campaigns['campaign_start'].dt.dayofweek

# Baseline donation rate (donations-per-day in the 30 days BEFORE the campaign)
def baseline_rate(campaign_start, lookback=30):
    if pd.isnull(campaign_start):
        return 0.0
    window_start = campaign_start - pd.Timedelta(days=lookback)
    mask = (
        (donations['don_date_parsed'] >= window_start) &
        (donations['don_date_parsed'] < campaign_start)
    )
    return float(donations[mask]['amount_num'].sum()) / lookback

campaigns['pre_campaign_baseline'] = [
    baseline_rate(r['campaign_start']) for _, r in campaigns.iterrows()
]

# Revenue lift over baseline
expected_baseline = campaigns['pre_campaign_baseline'] * CAMPAIGN_WINDOW_DAYS
campaigns['revenue_lift'] = campaigns['campaign_revenue'] - expected_baseline

print('Seasonality features added.')
print(campaigns[['campaign_id','campaign_month','is_holiday_season',
                  'pre_campaign_baseline','revenue_lift']].head(8))

In [ ]:
# ── Exploratory Data Analysis ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Campaign revenue distribution
axes[0, 0].hist(campaigns['campaign_revenue'], bins=15, edgecolor='white')
axes[0, 0].set_title('Campaign Revenue Distribution')
axes[0, 0].set_xlabel('Total Attributed Revenue ($)')

# 2. Revenue vs. number of posts
axes[0, 1].scatter(campaigns['num_posts'], campaigns['campaign_revenue'],
                   alpha=0.7, s=50, color='steelblue')
axes[0, 1].set_title('Revenue vs. Number of Posts')
axes[0, 1].set_xlabel('Posts in Campaign')
axes[0, 1].set_ylabel('Revenue ($)')

# 3. Revenue by dominant platform
plat_rev = campaigns.groupby('dominant_platform')['campaign_revenue'].mean().sort_values(ascending=False)
plat_rev.plot(kind='bar', ax=axes[0, 2], color='coral', edgecolor='white')
axes[0, 2].set_title('Avg Revenue by Dominant Platform')
axes[0, 2].set_xticklabels(axes[0, 2].get_xticklabels(), rotation=30, ha='right')

# 4. Revenue by month
month_rev = campaigns.groupby('campaign_month')['campaign_revenue'].mean()
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
month_rev.index = [month_names.get(m, str(m)) for m in month_rev.index]
month_rev.plot(kind='bar', ax=axes[1, 0], color='mediumseagreen', edgecolor='white')
axes[1, 0].set_title('Avg Revenue by Campaign Month')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=30)

# 5. Revenue vs. avg engagement
axes[1, 1].scatter(campaigns['avg_engagement'], campaigns['campaign_revenue'],
                   alpha=0.7, s=50, color='orchid')
axes[1, 1].set_title('Revenue vs. Avg Post Engagement')
axes[1, 1].set_xlabel('Avg Engagement per Post')
axes[1, 1].set_ylabel('Revenue ($)')

# 6. New donors vs. total revenue
axes[1, 2].scatter(campaigns['new_donors_acquired'], campaigns['campaign_revenue'],
                   alpha=0.7, s=50, color='darkorange')
axes[1, 2].set_title('New Donors vs. Revenue')
axes[1, 2].set_xlabel('New Donors Acquired')
axes[1, 2].set_ylabel('Revenue ($)')

plt.suptitle('Campaign Effectiveness EDA — BrightHut', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
numeric_feature_cols = [
    'campaign_revenue', 'num_posts', 'num_platforms', 'campaign_duration_days',
    'avg_engagement', 'total_engagement', 'pct_cta_posts', 'pct_impact_posts',
    'pct_amount_posts', 'has_matching_post', 'pct_video_posts', 'pct_image_posts',
    'avg_caption_length', 'avg_hashtags', 'is_holiday_season', 'is_year_end',
    'is_summer', 'pre_campaign_baseline', 'new_donors_acquired'
]
avail = [c for c in numeric_feature_cols if c in campaigns.columns]
corr = campaigns[avail].corr()

plt.figure(figsize=(13, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, annot_kws={'size': 7})
plt.title('Campaign-Level Feature Correlation Matrix')
plt.tight_layout()
plt.show()

---
## 3. Modeling & Feature Selection

In [ ]:
# ── Assemble feature matrix ───────────────────────────────────────────────────

# Platform dummies
plat_dummies = pd.get_dummies(campaigns['dominant_platform'], prefix='plat',
                              drop_first=False, dtype=int)
campaigns = pd.concat([campaigns, plat_dummies], axis=1)
platform_cols = list(plat_dummies.columns)

BASE_FEATURES = [
    # Campaign volume and structure
    'num_posts', 'num_platforms', 'campaign_duration_days',
    # Engagement quality
    'avg_engagement', 'max_engagement', 'total_engagement',
    # Content strategy
    'pct_cta_posts', 'pct_impact_posts', 'pct_amount_posts',
    'has_matching_post', 'pct_video_posts', 'pct_image_posts',
    'avg_caption_length', 'avg_hashtags',
    # Timing / seasonality
    'campaign_month', 'is_holiday_season', 'is_year_end',
    'is_giving_tuesday', 'is_summer', 'day_of_week_start',
    # Baseline context
    'pre_campaign_baseline',
]

ALL_FEATURES = BASE_FEATURES + platform_cols
ALL_FEATURES = [c for c in ALL_FEATURES if c in campaigns.columns]

# Fill any remaining NaN
campaigns[ALL_FEATURES] = campaigns[ALL_FEATURES].fillna(0)

# Outcome variables
campaigns['log_campaign_revenue'] = np.log1p(campaigns['campaign_revenue'])

rev_thresh = campaigns['campaign_revenue'].quantile(0.60)
campaigns['high_revenue_campaign'] = (campaigns['campaign_revenue'] >= rev_thresh).astype(int)

X = campaigns[ALL_FEATURES].copy()
y = campaigns['high_revenue_campaign'].copy()

print(f'Feature matrix: {X.shape}')
print(f'High-revenue campaigns: {y.mean():.2%}  ({y.sum()} / {len(y)})')
print(f'Revenue 60th percentile threshold: ${rev_thresh:,.0f}')

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
# Chronological split: train on earlier campaigns, test on later ones.
# Random splits would leak future temporal context into training.

campaigns_sorted = campaigns.sort_values('campaign_start').reset_index(drop=True)
X_sorted = campaigns_sorted[ALL_FEATURES].fillna(0)
y_sorted = campaigns_sorted['high_revenue_campaign']

split_idx = max(1, int(len(campaigns_sorted) * 0.75))
X_train, X_test = X_sorted.iloc[:split_idx], X_sorted.iloc[split_idx:]
y_train, y_test = y_sorted.iloc[:split_idx], y_sorted.iloc[split_idx:]

print(f'Train: {len(X_train)} campaigns  |  Test: {len(X_test)} campaigns')

# Cross-validation (StratifiedKFold on train set — for class balance)
cv_n = min(5, max(2, int(len(X_train) / 2)))
cv   = StratifiedKFold(n_splits=cv_n, shuffle=True, random_state=RANDOM_STATE)
print(f'CV folds: {cv_n}')

# ── Small-sample notice ───────────────────────────────────────────────────────
if len(campaigns_sorted) < 20:
    print()
    print('⚠ NOTE: Small campaign sample detected (<20 campaigns).')
    print('  ML estimates will be directional/hypothesis-generating, not definitive.')
    print('  The OLS explanatory model will be the primary analytical output.')

In [ ]:
# ── Model A: Logistic Regression (baseline classifier) ───────────────────────

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
                   class_weight='balanced', max_iter=1000,
                   C=0.5, random_state=RANDOM_STATE
               ))
])

if len(X_train) >= cv_n * 2 and y_train.nunique() > 1:
    cv_lr = cross_val_score(pipe_lr, X_train, y_train, cv=cv,
                            scoring='roc_auc', n_jobs=-1)
    print(f'LR  CV ROC-AUC: {cv_lr.mean():.4f}  (±{cv_lr.std():.4f})')
else:
    print('LR  CV skipped — insufficient samples for cross-validation')
pipe_lr.fit(X_train, y_train)

In [ ]:
# ── Model B: Random Forest ────────────────────────────────────────────────────

pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    RandomForestClassifier(
                   n_estimators=300, max_depth=4,
                   min_samples_leaf=2, class_weight='balanced',
                   random_state=RANDOM_STATE, n_jobs=-1
               ))
])

if len(X_train) >= cv_n * 2 and y_train.nunique() > 1:
    cv_rf = cross_val_score(pipe_rf, X_train, y_train, cv=cv,
                            scoring='roc_auc', n_jobs=-1)
    print(f'RF  CV ROC-AUC: {cv_rf.mean():.4f}  (±{cv_rf.std():.4f})')
else:
    print('RF  CV skipped — insufficient samples')
pipe_rf.fit(X_train, y_train)

In [ ]:
# ── Model C: Gradient Boosted Trees (primary predictive model) ────────────────

pipe_gbt = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    GradientBoostingClassifier(
                   n_estimators=200, learning_rate=0.05,
                   max_depth=3, subsample=0.8,
                   random_state=RANDOM_STATE
               ))
])

if len(X_train) >= cv_n * 2 and y_train.nunique() > 1:
    cv_gbt = cross_val_score(pipe_gbt, X_train, y_train, cv=cv,
                             scoring='roc_auc', n_jobs=-1)
    print(f'GBT CV ROC-AUC: {cv_gbt.mean():.4f}  (±{cv_gbt.std():.4f})')
else:
    print('GBT CV skipped — insufficient samples')
pipe_gbt.fit(X_train, y_train)

---
## 4. Evaluation & Selection

In [ ]:
# ── Hold-out test performance ─────────────────────────────────────────────────
results = {}
for name, pipe in [('Logistic Regression', pipe_lr),
                   ('Random Forest',       pipe_rf),
                   ('Gradient Boosting',   pipe_gbt)]:
    if len(X_test) == 0 or y_test.nunique() < 2:
        print(f'{name:25s}  Test set too small for AUC — reporting train AUC')
        proba_tr = pipe.predict_proba(X_train)[:, 1]
        auc = roc_auc_score(y_train, proba_tr) if y_train.nunique() > 1 else float('nan')
        results[name] = {'ROC-AUC': round(auc, 4), 'proba': pipe.predict_proba(X_train)[:, 1]}
    else:
        proba = pipe.predict_proba(X_test)[:, 1]
        auc   = roc_auc_score(y_test, proba) if y_test.nunique() > 1 else float('nan')
        results[name] = {'ROC-AUC': round(auc, 4), 'proba': proba}
    print(f'{name:25s}  ROC-AUC = {results[name]["ROC-AUC"]:.4f}')

best_name = max(results, key=lambda k: results[k]['ROC-AUC'])
best_pipe = {'Logistic Regression': pipe_lr,
             'Random Forest':       pipe_rf,
             'Gradient Boosting':   pipe_gbt}[best_name]
print(f'\nBest model: {best_name}')

In [ ]:
# ── Threshold tuning (F1 balanced — campaign planning is symmetric) ───────────
# For campaign planning we weight precision and recall equally (F1, beta=1).
# The cost of a false positive (running a planned campaign that underperforms)
# is similar to a false negative (canceling a campaign that would have succeeded).

eval_X = X_test if len(X_test) > 0 else X_train
eval_y = y_test if len(y_test) > 0 else y_train

if eval_y.nunique() > 1:
    best_proba = best_pipe.predict_proba(eval_X)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(eval_y, best_proba)
    f1_scores = [
        fbeta_score(eval_y, (best_proba >= t).astype(int), beta=1.0)
        for t in thresholds
    ]
    best_idx       = int(np.argmax(f1_scores))
    BEST_THRESHOLD = float(thresholds[best_idx])

    print(f'Optimal threshold (max F1): {BEST_THRESHOLD:.3f}')
    print(f'  Precision: {precisions[best_idx]:.3f}')
    print(f'  Recall:    {recalls[best_idx]:.3f}')
    print(f'  F1:        {f1_scores[best_idx]:.3f}')

    plt.figure(figsize=(8, 4))
    plt.plot(thresholds, f1_scores, color='teal')
    plt.axvline(BEST_THRESHOLD, color='red', linestyle='--',
                label=f'Best threshold = {BEST_THRESHOLD:.3f}')
    plt.xlabel('Decision Threshold')
    plt.ylabel('F1 Score')
    plt.title('F1-Optimized Threshold — Campaign Revenue Classifier')
    plt.legend()
    plt.tight_layout()
    plt.show()

    y_pred = (best_proba >= BEST_THRESHOLD).astype(int)
    print('\nClassification Report:')
    print(classification_report(eval_y, y_pred,
                                target_names=['Below-Avg Campaign', 'High-Revenue Campaign']))
else:
    BEST_THRESHOLD = 0.5
    print('Skipping threshold tuning — evaluation set has only one class.')

In [ ]:
# ── Permutation importance on evaluation set ──────────────────────────────────

if eval_y.nunique() > 1 and len(eval_X) >= 4:
    perm = permutation_importance(
        best_pipe, eval_X, eval_y,
        n_repeats=20, scoring='roc_auc',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    perm_df = pd.DataFrame({
        'feature':    ALL_FEATURES,
        'importance': perm.importances_mean,
        'std':        perm.importances_std
    }).sort_values('importance', ascending=False).head(20)

    plt.figure(figsize=(9, 6))
    plt.barh(perm_df['feature'][::-1], perm_df['importance'][::-1],
             xerr=perm_df['std'][::-1], color='darkorange', alpha=0.85)
    plt.xlabel('Mean Decrease in ROC-AUC')
    plt.title(f'Feature Importances (Permutation) — {best_name}')
    plt.tight_layout()
    plt.show()

    print('Top 10 features by permutation importance:')
    print(perm_df.head(10).to_string(index=False))
else:
    print('Permutation importance skipped — evaluation set too small or single-class.')

---
## 5. Causal and Relationship Analysis

### 5a. OLS Regression on Log-Campaign-Revenue (Explanatory — Primary)

The OLS model is the **primary analytical output** of this pipeline. Unlike the GBT/RF ensemble, OLS produces effect size estimates with confidence intervals and p-values that answer the organization's actual question: *"Holding other attributes constant, what is the marginal contribution of each campaign design decision to revenue?"

**Features excluded from the explanatory model (tautological / post-campaign):**
- `total_engagement` — depends on how many posts ran (collinear with `num_posts`); use `avg_engagement` instead
- `max_engagement` — a single outlier post inflates this; `avg_engagement` is more stable
- `campaign_revenue` / `log_campaign_revenue` — the outcome itself
- `high_revenue_campaign` — derived from the outcome
- `revenue_lift` — derived from the outcome
- `new_donors_acquired` — an alternative outcome, not a cause

All features are standardized before OLS so coefficients are comparable across variables with different scales.

In [ ]:
# ── OLS on log_campaign_revenue ───────────────────────────────────────────────

EXCLUDE_OLS = {
    'campaign_revenue', 'log_campaign_revenue', 'high_revenue_campaign',
    'revenue_lift', 'new_donors_acquired', 'total_engagement', 'max_engagement'
}
ols_features = [c for c in ALL_FEATURES if c not in EXCLUDE_OLS]

y_ols = campaigns['log_campaign_revenue']
X_ols = campaigns[ols_features].copy().fillna(0)

# Standardize for coefficient comparability
X_ols_std = (X_ols - X_ols.mean()) / (X_ols.std().replace(0, 1))
X_ols_c   = sm.add_constant(X_ols_std)

if len(campaigns) >= len(ols_features) + 2:
    ols_model = sm.OLS(y_ols, X_ols_c).fit()
    print(ols_model.summary())
else:
    # More features than observations: use Ridge regression instead
    print('⚠ Fewer campaigns than features — switching to Ridge regression for stability.')
    ridge = Ridge(alpha=1.0, fit_intercept=True)
    ridge.fit(X_ols_std, y_ols)
    coef_df_r = pd.DataFrame({'feature': ols_features, 'ridge_coef': ridge.coef_})\
                  .sort_values('ridge_coef', key=abs, ascending=False)
    print('Ridge coefficients (standardized features):')
    print(coef_df_r.to_string(index=False))
    ols_model = None

In [ ]:
# ── Visualize OLS / Ridge coefficients ───────────────────────────────────────

if ols_model is not None:
    coef_df = pd.DataFrame({
        'feature': ols_model.params.index,
        'coef':    ols_model.params.values,
        'ci_lo':   ols_model.conf_int()[0].values,
        'ci_hi':   ols_model.conf_int()[1].values,
        'pvalue':  ols_model.pvalues.values
    }).query("feature != 'const'")
    coef_df['abs_coef'] = coef_df['coef'].abs()
    coef_df = coef_df.sort_values('abs_coef', ascending=False).head(20)

    colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['coef']]
    plt.figure(figsize=(10, 7))
    plt.barh(coef_df['feature'][::-1], coef_df['coef'][::-1],
             color=colors[::-1], alpha=0.85)
    plt.errorbar(
        coef_df['coef'][::-1],
        range(len(coef_df)),
        xerr=[
            (coef_df['coef'] - coef_df['ci_lo'])[::-1].values,
            (coef_df['ci_hi'] - coef_df['coef'])[::-1].values
        ],
        fmt='none', color='black', capsize=3, linewidth=1
    )
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Standardized OLS Coefficient (effect on log-revenue)')
    plt.title('Campaign Effectiveness: Feature Effects on Revenue\n(Standardized — comparable across features)')
    plt.tight_layout()
    plt.show()

    print('\nStatistically significant features (p < 0.10):')
    sig = coef_df[coef_df['pvalue'] < 0.10].sort_values('coef', ascending=False)
    print(sig[['feature','coef','pvalue']].to_string(index=False))
    print(f'\nModel adj. R²: {ols_model.rsquared_adj:.4f}')
    print(f'Model AIC: {ols_model.aic:.1f}')

In [ ]:
# ── 5b. New donor acquisition analysis ───────────────────────────────────────
# Are campaigns generating new donors, or only re-engaging existing ones?
# This matters: new donors represent long-term pipeline growth, not just one-time revenue.

print('=== Campaign Revenue Composition ===')
print(f'  Campaigns with at least 1 new donor: '
      f'{(campaigns["new_donors_acquired"] > 0).sum()} / {len(campaigns)}')
print(f'  Avg new donors per campaign: {campaigns["new_donors_acquired"].mean():.1f}')
print(f'  Max new donors in one campaign: {campaigns["new_donors_acquired"].max()}')

# Do holiday campaigns acquire more new donors?
if 'is_holiday_season' in campaigns.columns:
    holiday_new = campaigns.groupby('is_holiday_season')['new_donors_acquired'].mean()
    print('\n  Avg new donors by holiday season:')
    for k, v in holiday_new.items():
        print(f'    is_holiday={k}: {v:.2f}')

# Correlation between new donors and total revenue
corr_nd = campaigns['new_donors_acquired'].corr(campaigns['campaign_revenue'])
print(f'\n  Correlation: new_donors_acquired vs campaign_revenue = {corr_nd:.3f}')

In [ ]:
# ── 5c. Revenue lift by key campaign attributes ───────────────────────────────
# Actionable comparisons: what is the revenue difference for campaigns with vs.
# without each key attribute?

BINARY_ATTRS = [
    ('is_holiday_season',   'Holiday Season'),
    ('has_matching_post',   'Matching Gift Mentioned'),
    ('is_year_end',         'Year-End Campaign'),
    ('is_giving_tuesday',   'Giving Tuesday Window'),
]

print('Revenue comparison by binary campaign attribute:')
print(f"{'Attribute':35s}  {'With ($)':>10s}  {'Without ($)':>12s}  {'Lift ($)':>10s}")
print('-' * 75)
for col, label in BINARY_ATTRS:
    if col not in campaigns.columns:
        continue
    rev_with    = campaigns[campaigns[col] == 1]['campaign_revenue'].mean()
    rev_without = campaigns[campaigns[col] == 0]['campaign_revenue'].mean()
    lift        = rev_with - rev_without
    print(f"{label:35s}  {rev_with:>10,.0f}  {rev_without:>12,.0f}  {lift:>+10,.0f}")

# Continuous attribute: does more posts per campaign always help?
print('\nRevenue by campaign size (quartiles of num_posts):')
campaigns['posts_quartile'] = pd.qcut(campaigns['num_posts'], q=4, labels=['Q1','Q2','Q3','Q4'],
                                       duplicates='drop')
print(campaigns.groupby('posts_quartile')['campaign_revenue'].mean().round(0))

# Engagement quality vs. quantity
print('\nCorrelation: avg_engagement vs campaign_revenue =',
      campaigns['avg_engagement'].corr(campaigns['campaign_revenue']).round(3))
print('Correlation: num_posts vs campaign_revenue =',
      campaigns['num_posts'].corr(campaigns['campaign_revenue']).round(3))

---
## 6. Deployment Notes

### What to Deploy

Two artifacts are produced:

1. **Campaign effectiveness classifier** (`campaign_effectiveness_model.pkl`) — The best-performing predictive model, used to score planned campaigns before launch.
2. **Campaign config** (`campaign_effectiveness_config.json`) — Stores the operating threshold, feature list, OLS coefficient estimates for the recommendation engine, and metadata.

### Integration Points

| Deployment Target | Description |
|---|---|
| `GET /api/ml/campaign-effectiveness` | Admin-only endpoint; accepts campaign metadata and returns predicted revenue tier + coefficient-based recommendations |
| Campaign Planning Dashboard | Staff enter proposed campaign attributes; system returns "predicted: HIGH/LOW revenue" with top 3 recommendation adjustments |
| Reports & Analytics Page | Historical campaign performance table with model scores; OLS coefficient chart as a permanent "what works" reference |
| Pre-Launch Campaign Checklist | Inline scoring widget on the campaign creation form (similar to post-engagement checker) |

### Retraining Schedule

- **Quarterly retraining** recommended as new campaigns accumulate
- With a small initial dataset, prioritize the OLS explanatory coefficients as the primary decision guide
- As the campaign catalog grows (20+ campaigns), the predictive model becomes more reliable

### Business Recommendations from Explanatory Model

Expected findings that the OLS model should surface, based on nonprofit fundraising research and the data design:

- **Seasonality dominates:** Year-end and holiday-season campaigns typically generate 2–4× the revenue of off-season campaigns. If the org can only run 2–3 campaigns per year, November–December is where to focus.
- **Matching gifts amplify revenue:** Even a small matching commitment from a major donor signals urgency and social proof. The coefficient on `has_matching_post` is expected to be strongly positive.
- **Impact messaging converts:** Posts that name specific outcomes ("14 girls received counseling", "3 residents completed school") are expected to outperform generic fundraising asks.
- **Quality over quantity:** A smaller number of high-engagement posts likely outperforms a large volume of low-engagement content. `avg_engagement` is expected to have a stronger effect than `num_posts`.
- **Platform matters:** If Facebook or Instagram significantly outperforms LinkedIn for this audience, the platform coefficient will surface that and allow the org to reallocate effort.

⚠️ **Causality caution:** OLS coefficients are association estimates, not causal effects. Confounders (economic conditions, news events, donor base maturation) are not in the model. Use coefficients as hypotheses to test through planned variation in future campaigns, not as rules to follow blindly.

In [ ]:
# ── Save model artifacts ──────────────────────────────────────────────────────
ARTIFACT_DIR = Path('model_artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)

model_path  = ARTIFACT_DIR / 'campaign_effectiveness_model.pkl'
config_path = ARTIFACT_DIR / 'campaign_effectiveness_config.json'

joblib.dump(best_pipe, model_path)
print(f'Model saved → {model_path}')

# Extract OLS coefficients for the recommendation engine (if available)
ols_coefs = {}
if ols_model is not None:
    ols_coefs = {
        feat: {'coef': round(float(c), 6), 'pvalue': round(float(p), 4)}
        for feat, c, p in zip(
            ols_model.params.index, ols_model.params.values, ols_model.pvalues.values
        ) if feat != 'const'
    }

config = {
    'model_type':            best_name,
    'threshold':             round(BEST_THRESHOLD, 4),
    'feature_cols':          ALL_FEATURES,
    'ols_features':          ols_features,
    'ols_coefficients':      ols_coefs,
    'revenue_threshold_60p': round(float(rev_thresh), 2),
    'campaign_window_days':  CAMPAIGN_WINDOW_DAYS,
    'target':                'high_revenue_campaign',
    'notes':                 ('Campaigns attributed using a {}-day window. '
                              'Retrain quarterly as new campaigns accumulate.'.format(CAMPAIGN_WINDOW_DAYS)),
    'api_endpoint':          'GET /api/ml/campaign-effectiveness',
    'auth_required':         'admin'
}

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'Config saved → {config_path}')

In [ ]:
# ── Inference function for API integration ────────────────────────────────────

def score_campaign(campaign_metadata: dict) -> dict:
    """
    Score a planned campaign configuration before launch.

    Parameters
    ----------
    campaign_metadata : dict
        Keys should match ALL_FEATURES. Missing keys default to 0.
        Expected keys include: num_posts, num_platforms, campaign_duration_days,
        avg_engagement (leave 0 for planned campaigns), pct_cta_posts,
        pct_impact_posts, has_matching_post, pct_video_posts, pct_image_posts,
        is_holiday_season, is_year_end, campaign_month, pre_campaign_baseline.

    Returns
    -------
    dict with keys:
        - revenue_score        : float  probability of high-revenue campaign (0–1)
        - predicted_tier       : str    'HIGH' or 'LOW'
        - top_recommendations  : list   top 3 attribute adjustments by OLS coefficient
        - ols_adj_r2           : float  model explanatory power (if OLS was fit)
    """
    with open(config_path) as f:
        cfg = json.load(f)

    model = joblib.load(model_path)
    row   = {col: campaign_metadata.get(col, 0) for col in cfg['feature_cols']}
    X_new = pd.DataFrame([row])

    score = float(model.predict_proba(X_new)[0, 1])
    tier  = 'HIGH' if score >= cfg['threshold'] else 'LOW'

    # Generate top recommendations from OLS coefficients
    recommendations = []
    if cfg.get('ols_coefficients'):
        coefs = cfg['ols_coefficients']
        # For features the campaign is NOT using (value == 0), recommend the ones
        # with the largest positive OLS coefficient
        unused_positive = [
            (feat, info['coef'])
            for feat, info in coefs.items()
            if info['coef'] > 0 and campaign_metadata.get(feat, 0) == 0
        ]
        unused_positive.sort(key=lambda x: x[1], reverse=True)
        recommendations = [
            f"Consider adding '{feat}' (OLS coef: +{coef:.3f} on log-revenue)"
            for feat, coef in unused_positive[:3]
        ]

    return {
        'revenue_score':       round(score, 4),
        'predicted_tier':      tier,
        'top_recommendations': recommendations,
        'ols_adj_r2':          ols_model.rsquared_adj if ols_model is not None else None
    }


# ── Example usage ─────────────────────────────────────────────────────────────
example_campaign = {
    'num_posts':            8,
    'num_platforms':        2,
    'campaign_duration_days': 10,
    'pct_cta_posts':        0.75,
    'pct_impact_posts':     0.50,
    'has_matching_post':    1,
    'pct_video_posts':      0.25,
    'pct_image_posts':      0.50,
    'is_holiday_season':    1,
    'is_year_end':          1,
    'campaign_month':       12,
    'avg_hashtags':         4,
}

result = score_campaign(example_campaign)
print('\nExample campaign score:')
for k, v in result.items():
    print(f'  {k}: {v}')